# Synthetic V-MD3 Range–Angle and SAR Processing from First Principles

This notebook builds a complete synthetic FMCW data set with dimensions chosen
to resemble the V-MD3 measurements, then follows the data through three
products:

1. a conventional four-RX **range–angle heatmap** at one rail position;
2. the **unfocused range–aperture data** from all rail positions; and
3. a **focused SAR image** formed by backprojection.

The same complex range profiles feed both spatial-processing paths. The point
where they split—and the aperture used by each path—is made explicit.

The synthetic scene contains two ideal point scatterers separated by
$0.30\ \mathrm{m}$ in azimuth/cross range and $0.50\ \mathrm{m}$ in downrange.
A long synthetic aperture produces a fine-resolution image and makes their
curved range histories easy to see before focusing.

> **Standalone notebook:** No external data file is required. Section 4
> generates the complete synthetic raw-I/Q tensor from the target table and
> FMCW parameters. Edit the configuration cell in Section 3 and choose
> **Run All** to generate a new scene.

## Learning objectives

After working through the notebook, you should be able to explain:

- what each raw-data index represents;
- why a range FFT produces a **complex range profile**, not yet an image;
- why chirp and frame integration must preserve complex phase;
- how a physical-array steering vector is populated from RX-element positions;
- why a center-position range–angle heatmap is not SAR;
- why parabolic range histories appear in the unfocused range–aperture display,
  rather than in a single-position range–angle heatmap;
- how range interpolation and carrier-phase compensation perform SAR focusing;
- why aperture windowing changes sidelobes but does not create the focus; and
- how the equations map directly onto the Python operations.

## 1. Notation and data-axis convention

The full synthetic data set is stored as

$$
\boxed{x[a,f,p,m,n]}
$$

in the order

$$
\boxed{
\text{aperture position}
\times
\text{frame}
\times
\text{RX channel}
\times
\text{chirp}
\times
\text{fast-time sample}.
}
$$

| Index | Meaning |
|---|---|
| $a$ | rail/aperture-position index |
| $f$ | repeated-frame index at one fixed rail position |
| $p$ | RX-channel index |
| $m$ | chirp index within one frame |
| $n$ | fast-time ADC-sample index within one chirp |
| $k$ | range-bin index after the fast-time FFT |
| $u$ | candidate-angle index, corresponding to $\theta_u$ |
| $i,j$ | focused-image indices, corresponding to $(x_i,y_j)$ |

For one particular frame, the notation reduces to the four-index convention

$$
x_f[a,p,m,n].
$$

The letter order does not need to be alphabetical. Each index is chosen to
identify a physical or processing dimension.

### Conjugate-transpose notation

This notebook uses the physics dagger notation:

$$
\boxed{
\mathbf V^\dagger
\equiv
\mathbf V^H
\;=\;
(\mathbf V^*)^T.
}
$$

In NumPy, the corresponding operation is `V.conj().T`.

> **What to say:** The rail position $a$ is the synthetic-aperture dimension.
> The chirp index $m$ labels repeated FMCW chirps while the radar is stopped at
> one position. The frame index $f$ labels repeated 64-chirp acquisitions. This
> nested structure is why our laboratory notation needs both $a$ and $m$.

## 2. Processing overview

The range–angle and SAR paths share the same raw-data and range-processing
steps. They split only after the data have been reduced to one complex range
profile for every aperture position and RX channel:

$$
\overline S[a,p,k].
$$

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from matplotlib.patches import FancyBboxPatch, FancyArrowPatch


plt.rcParams.update(
    {
        "figure.figsize": (10, 6),
        "axes.grid": True,
        "grid.alpha": 0.25,
        "font.size": 11,
    }
)


def normalized_magnitude_db(complex_data, floor_db=-60.0):
    '''Return normalized complex magnitude in decibels.'''
    magnitude = np.abs(complex_data)
    magnitude /= np.maximum(magnitude.max(), np.finfo(float).tiny)
    magnitude_db = 20.0 * np.log10(
        np.maximum(magnitude, np.finfo(float).tiny)
    )
    return np.maximum(magnitude_db, floor_db)


def interpolate_complex_profile(range_axis_m, complex_profile, query_range_m):
    '''Linearly interpolate real and imaginary parts at arbitrary ranges.'''
    query_shape = np.shape(query_range_m)
    query_flat = np.asarray(query_range_m).ravel()
    real_part = np.interp(
        query_flat,
        range_axis_m,
        np.real(complex_profile),
        left=0.0,
        right=0.0,
    )
    imag_part = np.interp(
        query_flat,
        range_axis_m,
        np.imag(complex_profile),
        left=0.0,
        right=0.0,
    )
    return (real_part + 1j * imag_part).reshape(query_shape)

In [ ]:
def draw_processing_flow():
    '''Draw the processing split without requiring a Mermaid extension.'''
    fig, ax = plt.subplots(figsize=(14, 10))
    ax.set_xlim(0, 14)
    ax.set_ylim(0, 13)
    ax.axis("off")

    def box(x, y, width, height, text, color):
        patch = FancyBboxPatch(
            (x, y),
            width,
            height,
            boxstyle="round,pad=0.04,rounding_size=0.12",
            facecolor=color,
            edgecolor="0.25",
            linewidth=1.2,
        )
        ax.add_patch(patch)
        ax.text(
            x + width / 2,
            y + height / 2,
            text,
            ha="center",
            va="center",
            fontsize=10,
        )
        return (x, y, width, height)

    def arrow(source, destination):
        sx, sy, sw, sh = source
        dx, dy, dw, dh = destination
        start = (sx + sw / 2, sy)
        end = (dx + dw / 2, dy + dh)
        ax.add_patch(
            FancyArrowPatch(
                start,
                end,
                arrowstyle="-|>",
                mutation_scale=14,
                linewidth=1.3,
                color="0.25",
            )
        )

    raw = box(4.4, 11.3, 5.2, 1.0, "Raw complex I/Q\n$x[a,f,p,m,n]$", "#dbeafe")
    rng = box(4.4, 9.6, 5.2, 1.0, "Range FFT over sample $n$\n$S[a,f,p,m,k]$", "#dbeafe")
    avg = box(4.4, 7.9, 5.2, 1.0, "Coherent integration over chirp $m$ and frame $f$\n$\\overline{S}[a,p,k]$", "#dbeafe")

    conventional = box(0.6, 5.7, 5.2, 1.1, "Conventional path\nselect one position $a=a_0$", "#dcfce7")
    sar = box(8.2, 5.7, 5.2, 1.1, "SAR path\nretain all positions $a$", "#ffedd5")

    beam = box(0.6, 3.7, 5.2, 1.1, "Test candidate angle $\\theta_u$\nand sum across RX channel $p$", "#dcfce7")
    range_ap = box(8.2, 3.7, 5.2, 1.1, "Display $|\\overline{S}[a,p_0,k]|$\nUnfocused range–aperture data", "#ffedd5")

    heatmap = box(0.6, 1.7, 5.2, 1.1, "Range–angle heatmap\n$|H[k,u]|$", "#dcfce7")
    bp = box(8.2, 1.7, 5.2, 1.1, "Predict range, interpolate, phase-correct,\nand sum across aperture positions", "#ffedd5")
    focused = box(8.2, 0.0, 5.2, 1.0, "Focused complex SAR image\n$I[i,j]$", "#fee2e2")

    arrow(raw, rng)
    arrow(rng, avg)
    arrow(avg, conventional)
    arrow(avg, sar)
    arrow(conventional, beam)
    arrow(beam, heatmap)
    arrow(sar, range_ap)
    arrow(range_ap, bp)
    arrow(bp, focused)

    ax.set_title("Common FMCW processing followed by two spatial-processing paths", fontsize=15)
    plt.show()


draw_processing_flow()

## 3. Synthetic V-MD3-like system and scene

The default synthetic radar uses:

- $121$ mechanical aperture positions over a $60\ \mathrm{cm}$ rail;
- $5$ repeated frames at every position;
- $4$ RX channels with approximately half-wavelength spacing;
- $64$ chirps per frame;
- $128$ complex fast-time samples per chirp;
- a $61\ \mathrm{GHz}$ carrier; and
- a range mapping of approximately $4.69\ \mathrm{cm}$ per unpadded FFT bin.

The targets are ideal point scatterers, not finite-diameter spheres. Point
scatterers are the cleanest way to expose the point-spread function and
resolution of the processing algorithm. A sphere model could be added later by
summing scattering points over a spherical surface and assigning an RCS model.

Target 1 and Target 2 differ in both azimuth and downrange. The long aperture
produces range migration of several range bins, so the range-versus-aperture
plot shows two visible curved histories.

The rail samples are spaced by $5\ \mathrm{mm}$—approximately one wavelength.
This dense sampling is deliberate. Aperture **length** controls the ideal
azimuth resolution, while aperture **sample spacing** controls spatial aliasing
and grating lobes over the imaged angular sector. A long but sparsely sampled
rail can have a narrow main lobe and still produce false repeated targets.

For clarity, the synthetic geometry places the transmitter and RX channel 0
at the rail coordinate $x_a$. The other RX elements have offsets $x_p=pd$.
This makes RX channel 0 an exactly monostatic channel for the first SAR
backprojection. Real V-MD3 processing should use measured or documented
TX/RX/virtual phase-center coordinates.

> **Teaching takeaway:** The values in the following configuration cell are
> the main controls for the example. Students can change the aperture length,
> number of positions, target coordinates, amplitudes, or noise level and then
> rerun the complete notebook. The synthetic data are regenerated automatically.

In [ ]:
# Physical constants and FMCW parameters
c0_m_per_s = 299_792_458.0
carrier_frequency_hz = 61.0e9
wavelength_m = c0_m_per_s / carrier_frequency_hz
wavenumber_rad_per_m = 2.0 * np.pi / wavelength_m

number_of_positions = 121
number_of_frames = 5
number_of_rx_channels = 4
number_of_chirps = 64
number_of_fast_time_samples = 128

sampling_frequency_hz = 2.0e6
unaliased_range_extent_m = 6.0
nominal_range_bin_spacing_m = (
    unaliased_range_extent_m / number_of_fast_time_samples
)

# Select the chirp slope so the 128-bin complex FFT spans approximately 0--6 m.
chirp_slope_hz_per_s = (
    c0_m_per_s
    * sampling_frequency_hz
    / (
        2.0
        * number_of_fast_time_samples
        * nominal_range_bin_spacing_m
    )
)
chirp_duration_s = number_of_fast_time_samples / sampling_frequency_hz
chirp_bandwidth_hz = chirp_slope_hz_per_s * chirp_duration_s
theoretical_range_resolution_m = c0_m_per_s / (2.0 * chirp_bandwidth_hz)

# Mechanical aperture and receive-array coordinates
aperture_positions_m = np.linspace(-0.30, 0.30, number_of_positions)
rx_element_spacing_m = 2.464e-3
rx_offsets_m = np.arange(number_of_rx_channels) * rx_element_spacing_m

# Two distinct ideal point targets.
targets = pd.DataFrame(
    {
        "name": ["Target 1", "Target 2"],
        "cross_range_m": [-0.150, 0.150],
        "downrange_m": [1.200, 1.700],
        # The farther target is given a larger reflectivity so both targets
        # remain visually prominent after geometric spreading.
        "relative_amplitude": [1.00, 1.65],
        "scattering_phase_rad": [0.30, -0.90],
    }
)

parameter_table = pd.DataFrame(
    {
        "quantity": [
            "Carrier frequency",
            "Wavelength",
            "Aperture length",
            "Mechanical aperture spacing",
            "RX spacing",
            "Chirps per frame",
            "Fast-time samples",
            "Chirp bandwidth",
            "Unpadded range-bin spacing",
            "Ideal focused azimuth resolution at mean target range",
        ],
        "value": [
            f"{carrier_frequency_hz/1e9:.1f} GHz",
            f"{wavelength_m*1e3:.3f} mm",
            f"{np.ptp(aperture_positions_m)*100:.1f} cm",
            f"{np.diff(aperture_positions_m).mean()*1e3:.2f} mm",
            f"{rx_element_spacing_m*1e3:.3f} mm",
            str(number_of_chirps),
            str(number_of_fast_time_samples),
            f"{chirp_bandwidth_hz/1e9:.2f} GHz",
            f"{nominal_range_bin_spacing_m*100:.2f} cm",
            f"{wavelength_m*targets['downrange_m'].mean()/(2*np.ptp(aperture_positions_m))*100:.2f} cm",
        ],
    }
)

print(parameter_table.to_string(index=False))
print()
print(targets.to_string(index=False))

In [ ]:
fig, ax = plt.subplots(figsize=(9, 6))
ax.plot(
    aperture_positions_m,
    np.zeros_like(aperture_positions_m),
    "o-",
    color="tab:blue",
    label=f"{number_of_positions} rail positions",
)

for _, target in targets.iterrows():
    ax.scatter(
        target["cross_range_m"],
        target["downrange_m"],
        s=140,
        marker="x",
        linewidth=3,
        label=target["name"],
    )

ax.set_xlabel("Azimuth / cross range, $x$ (m)")
ax.set_ylabel("Downrange, $y$ (m)")
ax.set_title("Synthetic rail geometry and two-target scene")
ax.set_xlim(-0.48, 0.48)
ax.set_ylim(-0.05, 1.90)
ax.set_aspect("equal")
ax.legend(loc="upper right")
plt.show()

## 4. Generate the raw complex FMCW data

For target $t$, transmitter position $x_a$, and RX offset $x_p$, the transmit
and receive path lengths are

$$
R_{\mathrm{TX},a,t}
\;=\;
\sqrt{(x_t-x_a)^2+y_t^2},
$$

$$
R_{\mathrm{RX},a,p,t}
\;=\;
\sqrt{(x_t-x_a-x_p)^2+y_t^2}.
$$

The total bistatic path is

$$
L_{a,p,t}=R_{\mathrm{TX},a,t}+R_{\mathrm{RX},a,p,t}.
$$

The equivalent range used by the FMCW beat frequency is

$$
R_{\mathrm{eq},a,p,t}=\frac{L_{a,p,t}}{2},
$$

and the beat frequency is

$$
f_{b,a,p,t}
\;=\;
\frac{2K R_{\mathrm{eq},a,p,t}}{c_0}
\;=\;
\frac{K L_{a,p,t}}{c_0},
$$

where $K$ is the FMCW chirp slope.

The synthetic dechirped signal contains both the beat-frequency progression
and the carrier-sensitive propagation phase:

$$
x_t[a,f,p,m,n]
\;=\;
\alpha_t
\exp\left(j2\pi f_{b,a,p,t}\frac{n}{f_s}\right)
\exp\left(-j\frac{2\pi}{\lambda}L_{a,p,t}\right).
$$

For RX channel 0, $L_{a,0,t}=2R_{a,t}$ and the phase becomes the familiar

$$
-\frac{4\pi}{\lambda}R_{a,t}.
$$

> **Teaching takeaway:** This code cell is the data source. It evaluates the
> modeled path length to each point scatterer, converts that path into an FMCW
> beat frequency and carrier phase, and adds complex receiver noise. No
> external measurement file is loaded.

In [ ]:
rng = np.random.default_rng(2026)

# Full stored order: position, frame, RX channel, chirp, fast-time sample.
raw_iq = (
    0.12
    / np.sqrt(2.0)
    * (
        rng.standard_normal(
            (
                number_of_positions,
                number_of_frames,
                number_of_rx_channels,
                number_of_chirps,
                number_of_fast_time_samples,
            )
        )
        + 1j
        * rng.standard_normal(
            (
                number_of_positions,
                number_of_frames,
                number_of_rx_channels,
                number_of_chirps,
                number_of_fast_time_samples,
            )
        )
    )
).astype(np.complex64)

fast_time_s = np.arange(number_of_fast_time_samples) / sampling_frequency_hz

# Small common phase jitter preserves high coherence while making the repeated
# chirps and frames non-identical.
common_phase_jitter_rad = rng.normal(
    loc=0.0,
    scale=0.010,
    size=(
        number_of_positions,
        number_of_frames,
        1,
        number_of_chirps,
        1,
    ),
)

for _, target in targets.iterrows():
    x_target_m = target["cross_range_m"]
    y_target_m = target["downrange_m"]

    transmit_range_m = np.sqrt(
        (x_target_m - aperture_positions_m) ** 2
        + y_target_m**2
    )[:, np.newaxis]

    absolute_rx_positions_m = (
        aperture_positions_m[:, np.newaxis]
        + rx_offsets_m[np.newaxis, :]
    )
    receive_range_m = np.sqrt(
        (x_target_m - absolute_rx_positions_m) ** 2
        + y_target_m**2
    )

    total_path_m = transmit_range_m + receive_range_m
    equivalent_range_m = total_path_m / 2.0
    beat_frequency_hz = chirp_slope_hz_per_s * total_path_m / c0_m_per_s

    target_fast_time_signal = (
        target["relative_amplitude"]
        / equivalent_range_m[:, :, np.newaxis] ** 2
        * np.exp(
            1j
            * (
                2.0
                * np.pi
                * beat_frequency_hz[:, :, np.newaxis]
                * fast_time_s[np.newaxis, np.newaxis, :]
                - 2.0
                * np.pi
                * total_path_m[:, :, np.newaxis]
                / wavelength_m
                + target["scattering_phase_rad"]
            )
        )
    )

    raw_iq += (
        target_fast_time_signal[:, np.newaxis, :, np.newaxis, :]
        * np.exp(1j * common_phase_jitter_rad)
    ).astype(np.complex64)

print("raw_iq shape:", raw_iq.shape)
print("Axis order: (position, frame, RX channel, chirp, fast-time sample)")
print(f"Memory used by raw_iq: {raw_iq.nbytes/1024**2:.1f} MiB")

### Inspect one raw chirp

The raw I/Q samples are complex voltages versus fast time. The beat tones are
present, but the horizontal axis has not yet been converted into range.

In [ ]:
center_position_index = int(np.argmin(np.abs(aperture_positions_m)))
example_frame_index = 0
example_rx_channel = 0
example_chirp_index = 0

example_chirp = raw_iq[
    center_position_index,
    example_frame_index,
    example_rx_channel,
    example_chirp_index,
    :,
]

fig, ax = plt.subplots(figsize=(11, 5))
ax.plot(fast_time_s * 1e6, np.real(example_chirp), label="I component")
ax.plot(fast_time_s * 1e6, np.imag(example_chirp), label="Q component", alpha=0.8)
ax.set_xlabel("Fast time within chirp (µs)")
ax.set_ylabel("Complex voltage (arbitrary units)")
ax.set_title("One synthetic raw I/Q chirp at the center rail position")
ax.legend()
plt.show()

## 5. Common range processing before the paths split

Apply a Hann window along the fast-time sample index $n$, then perform the
range FFT:

$$
S[a,f,p,m,k]
\;=\;
\sum_{n=0}^{N-1}
w_r[n]x[a,f,p,m,n]e^{-j2\pi kn/N}.
$$

For stationary targets, coherently average over chirps and repeated frames:

$$
\boxed{
\overline S[a,p,k]
\;=\;
\frac{1}{FM}
\sum_{f=0}^{F-1}
\sum_{m=0}^{M-1}
S[a,f,p,m,k].
}
$$

This is a complex average. We do **not** average $|S|$ before beamforming or
SAR focusing.

Each range-bin value is a complex coefficient:

$$
\overline S[a,p,k]
\;=\;
I[a,p,k]+jQ[a,p,k]
\;=\;
A[a,p,k]e^{j\phi[a,p,k]}.
$$

> **What to say:** The range FFT converts fast-time beat frequency into slant
> range, but the output is still complex. At this point we know where echo
> energy occurs in range, and we have preserved the phase needed for both
> physical-array beamforming and synthetic-aperture focusing.

In [ ]:
fast_time_window = np.hanning(number_of_fast_time_samples)

# Process one aperture position at a time. This follows the mathematical order
# FFT_n followed by coherent averaging over frame f and chirp m, while avoiding
# storage of a very large five-dimensional FFT tensor.
complex_range_profiles_full = np.empty(
    (
        number_of_positions,
        number_of_rx_channels,
        number_of_fast_time_samples,
    ),
    dtype=np.complex128,
)

for aperture_index in range(number_of_positions):
    # raw_iq[a] has shape (frame, RX channel, chirp, sample).
    position_range_fft = np.fft.fft(
        raw_iq[aperture_index]
        * fast_time_window[np.newaxis, np.newaxis, np.newaxis, :],
        axis=-1,
    )

    # Average complex values over frame axis 0 and chirp axis 2.
    complex_range_profiles_full[aperture_index] = np.mean(
        position_range_fft,
        axis=(0, 2),
    )

# Keep the positive-frequency half, which covers the 1--2 m scene.
number_of_display_range_bins = number_of_fast_time_samples // 2
complex_range_profiles = complex_range_profiles_full[
    :, :, :number_of_display_range_bins
]

range_axis_m = (
    np.arange(number_of_display_range_bins)
    * sampling_frequency_hz
    / number_of_fast_time_samples
    * c0_m_per_s
    / (2.0 * chirp_slope_hz_per_s)
)

print(
    "One position FFT shape before averaging:",
    position_range_fft.shape,
)
print("complex_range_profiles shape:", complex_range_profiles.shape)
print("Final axis order: (aperture position, RX channel, range bin)")

In [ ]:
fig, ax = plt.subplots(figsize=(11, 6))

for channel_index in range(number_of_rx_channels):
    profile_db = normalized_magnitude_db(
        complex_range_profiles[
            center_position_index,
            channel_index,
            :,
        ],
        floor_db=-60,
    )
    ax.plot(
        range_axis_m,
        profile_db,
        label=f"RX {channel_index}",
        alpha=0.8,
    )

for target_index, (_, target) in enumerate(targets.iterrows()):
    ax.axvline(
        target["downrange_m"],
        color=["black", "tab:gray"][target_index],
        linestyle="--",
        label=f"{target['name']} nominal range",
    )
ax.set_xlim(0.7, 2.2)
ax.set_ylim(-55, 3)
ax.set_xlabel("Slant range (m)")
ax.set_ylabel("Normalized complex magnitude (dB)")
ax.set_title("Complex range profiles at the center rail position")
ax.legend(ncols=2)
plt.show()

## 6. The processing split

At this point the common data product is

$$
\boxed{\overline S[a,p,k]},
$$

with dimensions

$$
\text{aperture position}\times\text{RX channel}\times\text{range bin}.
$$

The two paths now ask different spatial questions:

| Conventional heatmap | SAR |
|---|---|
| Select one position $a=a_0$ | Retain all positions $a$ |
| Sum across physical RX channels $p$ | Sum across mechanical aperture positions $a$ |
| Test candidate angle $\theta_u$ | Test candidate image pixel $(x_i,y_j)$ |
| Use a physical-array steering vector | Use predicted range and propagation phase |

> **What to say:** This is the fork in the flowgraph. Nothing about the range
> processing decides whether the result is conventional imaging or SAR. The
> distinction is the spatial aperture we combine after the complex range
> profiles have been formed.

# Part I — Conventional four-RX range–angle map

## 7. Forming the steering vector

The RX-element positions are $x_p=pd$. These positions have units of meters.
They are not the steering-vector entries.

For a far-field arrival angle $\theta$, the approximate receive-path
difference is

$$
\Delta R_p(\theta)\approx-x_p\sin\theta.
$$

With the signal convention used in this simulation, the predicted channel
phasor is

$$
\boxed{
v_p(\theta)
\;=\;
e^{jk_0x_p\sin\theta},
\qquad
k_0=\frac{2\pi}{\lambda}.
}
$$

The steering vector is

$$
\mathbf v(\theta)
\;=\;
\begin{bmatrix}
v_0(\theta)&v_1(\theta)&v_2(\theta)&v_3(\theta)
\end{bmatrix}^{T}.
$$

At broadside, $\theta=0$ and

$$
\mathbf v(0)
\;=\;
\begin{bmatrix}1&1&1&1\end{bmatrix}^{T}.
$$

For a collection of candidate angles $\theta_u$, the steering matrix is

$$
V[p,u]=v_p(\theta_u).
$$

> **Teaching takeaway:** The steering-vector entries are not antenna
> coordinates. The coordinates $x_p$ are inserted into the phase model. The
> resulting dimensionless complex phasors $v_p(\theta_u)$ predict what each
> channel should measure for a wave from candidate azimuth $\theta_u$.

In [ ]:
candidate_angle_deg = np.linspace(-35.0, 35.0, 701)
candidate_angle_rad = np.deg2rad(candidate_angle_deg)

# Steering matrix V[p,u]: RX channel x candidate angle.
steering_matrix = np.exp(
    1j
    * wavenumber_rad_per_m
    * rx_offsets_m[:, np.newaxis]
    * np.sin(candidate_angle_rad)[np.newaxis, :]
)

print("RX element offsets (mm):", np.round(rx_offsets_m * 1e3, 3))
print("steering_matrix shape:", steering_matrix.shape)
print("Broadside steering vector:")
print(np.round(steering_matrix[:, np.argmin(np.abs(candidate_angle_deg))], 4))

fig, ax = plt.subplots(figsize=(10, 5))
for channel_index in range(number_of_rx_channels):
    ax.plot(
        candidate_angle_deg,
        np.unwrap(np.angle(steering_matrix[channel_index, :])),
        label=f"RX {channel_index}",
    )
ax.set_xlabel("Candidate angle, $\\theta_u$ (degrees)")
ax.set_ylabel("Predicted channel phase (rad)")
ax.set_title("Quantities used to populate the four-RX steering matrix")
ax.legend(ncols=2)
plt.show()

## 8. Beamform at the center rail position

At one range bin, collect the four measured channel values into

$$
\mathbf s_{a_0}[k]
\;=\;
\begin{bmatrix}
\overline S[a_0,0,k]&
\overline S[a_0,1,k]&
\overline S[a_0,2,k]&
\overline S[a_0,3,k]
\end{bmatrix}^{T}.
$$

The candidate-angle response is the matched spatial inner product

$$
\boxed{
H[k,u]
\;=\;
\mathbf v^\dagger(\theta_u)\mathbf s_{a_0}[k].
}
$$

In matrix form,

$$
\boxed{
H=V^\dagger S_{a_0}.
}
$$

The dagger conjugates the predicted phase progression. When the tested angle
matches the measured angle, the four channel phasors align and add
constructively.

> **What to say:** The dagger is the matched-filter step in space. It removes
> the phase progression predicted for one trial azimuth. If that trial is
> correct, the corrected RX-channel samples point in the same complex-plane
> direction and add coherently.

In [ ]:
# S_center has shape channel x range.
S_center = complex_range_profiles[center_position_index, :, :]

# NumPy V.conj().T is the dagger V^†.
# Output H has shape candidate angle x range bin.
range_angle_complex = steering_matrix.conj().T @ S_center
range_angle_db = normalized_magnitude_db(range_angle_complex, floor_db=-50)

true_angles_deg = np.rad2deg(
    np.arctan2(
        targets["cross_range_m"].to_numpy(),
        targets["downrange_m"].to_numpy(),
    )
)

fig, ax = plt.subplots(figsize=(11, 7))
image = ax.imshow(
    range_angle_db.T,
    origin="lower",
    aspect="auto",
    extent=[
        candidate_angle_deg[0],
        candidate_angle_deg[-1],
        range_axis_m[0],
        range_axis_m[-1],
    ],
    cmap="viridis",
    vmin=-35,
    vmax=0,
)

for angle_deg, (_, target) in zip(true_angles_deg, targets.iterrows()):
    ax.plot(
        angle_deg,
        target["downrange_m"],
        marker="x",
        markersize=11,
        markeredgewidth=2.5,
        color="red",
    )

ax.set_xlim(-25, 25)
ax.set_ylim(0.9, 2.05)
ax.set_xlabel("Azimuth angle (degrees)")
ax.set_ylabel("Slant range (m)")
ax.set_title("Conventional four-RX range–angle heatmap at one rail position")
fig.colorbar(image, ax=ax, label="Normalized coherent magnitude (dB)")
plt.show()

### Interpretation of the conventional heatmap

The red markers show the true target directions. The four-element physical
array still has a broad azimuth main lobe, although the $0.50\ \mathrm{m}$
downrange separation makes the two targets easy to distinguish in range.

This map uses only one mechanical rail position. It is therefore a conventional
physical-array image, not SAR.

> **Teaching takeaway:** A point target produces a compact range–angle response,
> not a parabolic trace. The parabola appears only after we put measurements
> from many rail positions side by side, because target range changes with
> aperture position.

# Part II — Unfocused SAR data

## 9. Select one monostatic RX channel and retain all positions

For the clearest first SAR derivation, select RX channel 0. In the synthetic
geometry, its phase center is colocated with the transmitter at every rail
position:

$$
S_{\mathrm{ap}}[a,k]
\;=\;
\overline S[a,0,k].
$$

The remaining dimensions are

$$
\text{aperture position}\times\text{range bin}.
$$

Displaying

$$
\boxed{|S_{\mathrm{ap}}[a,k]|}
$$

produces the unfocused range–aperture data. This is sometimes loosely called
an "unfocused SAR image," but its horizontal coordinate is the radar
position—not yet the estimated target cross-range coordinate.

The targets were observed at many rail positions, so their energy extends
across the aperture. With the $0.60\ \mathrm{m}$ aperture, the range
change is now large enough to trace the expected curved histories

$$
R_a(x_t,y_t)=\sqrt{(x_t-x_a)^2+y_t^2}.
$$

Near closest approach, this becomes

$$
R_a\approx R_0+\frac{(x_a-x_t)^2}{2R_0},
$$

which is the visible parabolic approximation.

> **What to say:** The horizontal axis is azimuth aperture position, not target
> azimuth. Each column is a complex range profile collected from a different
> radar location. The bright curved tracks show how the measured range to each
> target changes as the radar moves.

In [ ]:
sar_rx_channel = 0
unfocused_range_aperture = complex_range_profiles[:, sar_rx_channel, :]
unfocused_range_aperture_db = normalized_magnitude_db(
    unfocused_range_aperture,
    floor_db=-50,
)

fig, axes = plt.subplots(1, 2, figsize=(15, 6), constrained_layout=True)

image = axes[0].imshow(
    unfocused_range_aperture_db.T,
    origin="lower",
    aspect="auto",
    extent=[
        aperture_positions_m[0],
        aperture_positions_m[-1],
        range_axis_m[0],
        range_axis_m[-1],
    ],
    cmap="magma",
    vmin=-35,
    vmax=0,
)
axes[0].set_xlim(aperture_positions_m[0], aperture_positions_m[-1])
axes[0].set_ylim(0.9, 2.05)
axes[0].set_xlabel("Azimuth aperture position, $x_a$ (m)")
axes[0].set_ylabel("Slant range (m)")
axes[0].set_title("Unfocused range–aperture magnitude")
fig.colorbar(image, ax=axes[0], label="Normalized magnitude (dB)")

# Overlay the exact monostatic range history of each target.
for target_index, (_, target) in enumerate(targets.iterrows()):
    predicted_history_m = np.sqrt(
        (
            target["cross_range_m"]
            - aperture_positions_m
        ) ** 2
        + target["downrange_m"] ** 2
    )
    axes[0].plot(
        aperture_positions_m,
        predicted_history_m,
        linestyle="--",
        linewidth=1.6,
        color=["cyan", "white"][target_index],
        label=f"{target['name']} predicted $R_a$",
    )

    # Extract a phase history near the target's closest-approach range. This
    # fixed-bin diagnostic is intentionally simple; backprojection later
    # follows the fractional range history instead.
    target_range_bin = int(
        np.argmin(
            np.abs(
                range_axis_m
                - target["downrange_m"]
            )
        )
    )
    target_aperture_history = unfocused_range_aperture[:, target_range_bin]
    axes[1].plot(
        aperture_positions_m,
        np.unwrap(np.angle(target_aperture_history)),
        "o-",
        markersize=3,
        label=target["name"],
    )

axes[0].legend(loc="upper center", fontsize=9)
axes[1].set_xlabel("Azimuth aperture position, $x_a$ (m)")
axes[1].set_ylabel("Unwrapped complex phase (rad)")
axes[1].set_title("Phase histories near the two target ranges")
axes[1].legend()

plt.show()

### What is unfocused here?

Range processing has already concentrated each target in slant range, but no
operation has yet aligned its measurements across the mechanical aperture.
The target therefore remains distributed across the positions from which it
was observed.

The phase-history plot shows why simply averaging all aperture positions is
not sufficient. The complex phase rotates as the path length changes, so an
uncorrected coherent sum can partially cancel.

A formal **unfocused SAR processor** coherently integrates only a short
subaperture over which the phase is approximately constant. The approximate
limit used in the lecture is

$$
L_u\approx\sqrt{\frac{R\lambda}{2}}.
$$

The displayed range–aperture matrix is the input to such processing and to the
fully focused processor that follows.

> **Teaching takeaway:** Range migration and phase history come from the same
> function $R_a$. The magnitude plot reveals only migration large enough to
> cross resolvable range bins. The complex phase can reveal much smaller path
> changes because one phase cycle corresponds to only $\lambda/2$ of
> monostatic range change.

In [ ]:
reference_range_m = targets["downrange_m"].mean()
unfocused_aperture_limit_m = np.sqrt(reference_range_m * wavelength_m / 2.0)
focused_resolution_m = (
    wavelength_m
    * reference_range_m
    / (2.0 * np.ptp(aperture_positions_m))
)

print(
    "Approximate unfocused coherent-aperture limit L_u: "
    f"{unfocused_aperture_limit_m*100:.2f} cm"
)
print(
    "Approximate fully focused cross-range resolution: "
    f"{focused_resolution_m*100:.2f} cm"
)
print(
    "True azimuth separation: "
    f"{np.ptp(targets['cross_range_m']):.2f} m"
)
print(
    "True downrange separation: "
    f"{np.ptp(targets['downrange_m']):.2f} m"
)

# Part III — Focused SAR by backprojection

## 10. Densify the range grid for interpolation

Backprojection evaluates each range profile at generally fractional range-bin
locations. Zero-padding the 128 measured samples before the FFT gives a denser
numerical range grid:

$$
N_{\mathrm{FFT}}=1024.
$$

This does **not** add bandwidth or improve the physical range resolution. It
only makes complex interpolation more accurate and the displayed image
smoother.

Because the FFT is linear, averaging the complex raw chirps and frames before
this oversampled FFT is equivalent to averaging their oversampled spectra.

In [ ]:
oversampled_range_fft_size = 1024

# Average complex raw data over frame f and chirp m while retaining a, p, n.
complex_fast_time_mean = np.mean(raw_iq, axis=(1, 3))

oversampled_range_profiles = np.fft.fft(
    complex_fast_time_mean
    * fast_time_window[np.newaxis, np.newaxis, :],
    n=oversampled_range_fft_size,
    axis=-1,
)

number_of_positive_oversampled_bins = oversampled_range_fft_size // 2
oversampled_range_profiles = oversampled_range_profiles[
    :, :, :number_of_positive_oversampled_bins
]

oversampled_range_axis_m = (
    np.arange(number_of_positive_oversampled_bins)
    * sampling_frequency_hz
    / oversampled_range_fft_size
    * c0_m_per_s
    / (2.0 * chirp_slope_hz_per_s)
)

print("Measured fast-time samples:", number_of_fast_time_samples)
print("Zero-padded FFT size:", oversampled_range_fft_size)
print(
    "Oversampled grid spacing: "
    f"{np.diff(oversampled_range_axis_m).mean()*100:.3f} cm"
)
print(
    "Physical range resolution remains approximately: "
    f"{theoretical_range_resolution_m*100:.3f} cm"
)

## 11. Backprojection mathematics

For candidate pixel $(x_i,y_j)$ and rail position $x_a$, the predicted
monostatic range for RX channel 0 is

$$
\boxed{
R_a^{(i,j)}
\;=\;
\sqrt{(x_i-x_a)^2+y_j^2}.
}
$$

Backprojection then performs three essential operations.

### 11.1 Follow the predicted range history

Interpolate the complex range profile at the predicted range:

$$
\widetilde S_a^{(i,j)}
\;=\;
\widetilde S_{\mathrm{ap}}
\left[a,R_a^{(i,j)}\right].
$$

This accounts for range-cell migration.

### 11.2 Remove the predicted propagation phase

The synthetic measurement contains

$$
e^{-j4\pi R_a^{(i,j)}/\lambda}.
$$

The matched phase correction is therefore

$$
e^{+j4\pi R_a^{(i,j)}/\lambda}.
$$

### 11.3 Weight and coherently sum the aperture

$$
\boxed{
I[i,j]
\;=\;
\sum_{a=0}^{A-1}
w_a[a]\,
\widetilde S_a^{(i,j)}
e^{+j4\pi R_a^{(i,j)}/\lambda}.
}
$$

The interpolation and phase correction perform the focusing. The optional
window $w_a[a]$ changes the main-lobe/sidelobe tradeoff but is not what makes
the image focused.

> **What to say:** Backprojection tests one spatial hypothesis at a time. If a
> trial pixel is correct, its predicted ranges tell us where to sample each
> complex range profile, and its predicted phase tells us how to rotate each
> sample before addition. A wrong pixel fails one or both tests and therefore
> sums less coherently.

In [ ]:
def backproject_single_rx(
    aperture_range_profiles,
    range_axis_m,
    aperture_positions_m,
    image_cross_range_m,
    image_downrange_m,
    wavelength_m,
    aperture_weights=None,
):
    '''Focused monostatic backprojection for one complex RX channel.'''
    number_of_positions = aperture_range_profiles.shape[0]

    if aperture_weights is None:
        aperture_weights = np.ones(number_of_positions)

    aperture_weights = np.asarray(aperture_weights, dtype=float)
    aperture_weights /= np.sum(aperture_weights)

    cross_range_grid_m, downrange_grid_m = np.meshgrid(
        image_cross_range_m,
        image_downrange_m,
    )
    focused_image = np.zeros(cross_range_grid_m.shape, dtype=np.complex128)

    for aperture_index, aperture_position_m in enumerate(aperture_positions_m):
        # Predicted slant range R_a^(i,j) for every trial image pixel.
        predicted_range_m = np.sqrt(
            (cross_range_grid_m - aperture_position_m) ** 2
            + downrange_grid_m**2
        )

        # Complex interpolation follows the candidate pixel's range history.
        interpolated_complex_data = interpolate_complex_profile(
            range_axis_m=range_axis_m,
            complex_profile=aperture_range_profiles[aperture_index, :],
            query_range_m=predicted_range_m,
        )

        # Matched carrier-phase correction for measured exp(-j 4πR/λ).
        phase_correction = np.exp(
            1j * 4.0 * np.pi * predicted_range_m / wavelength_m
        )

        focused_image += (
            aperture_weights[aperture_index]
            * interpolated_complex_data
            * phase_correction
        )

    return focused_image

In [ ]:
image_cross_range_m = np.linspace(-0.30, 0.30, 301)
image_downrange_m = np.linspace(0.90, 2.05, 401)

single_rx_oversampled_profiles = oversampled_range_profiles[:, sar_rx_channel, :]

uniform_aperture_weights = np.ones(number_of_positions)
hann_aperture_weights = np.hanning(number_of_positions)

focused_uniform = backproject_single_rx(
    aperture_range_profiles=single_rx_oversampled_profiles,
    range_axis_m=oversampled_range_axis_m,
    aperture_positions_m=aperture_positions_m,
    image_cross_range_m=image_cross_range_m,
    image_downrange_m=image_downrange_m,
    wavelength_m=wavelength_m,
    aperture_weights=uniform_aperture_weights,
)

focused_hann = backproject_single_rx(
    aperture_range_profiles=single_rx_oversampled_profiles,
    range_axis_m=oversampled_range_axis_m,
    aperture_positions_m=aperture_positions_m,
    image_cross_range_m=image_cross_range_m,
    image_downrange_m=image_downrange_m,
    wavelength_m=wavelength_m,
    aperture_weights=hann_aperture_weights,
)

focused_uniform_db = normalized_magnitude_db(focused_uniform, floor_db=-50)
focused_hann_db = normalized_magnitude_db(focused_hann, floor_db=-50)

fig, axes = plt.subplots(1, 2, figsize=(15, 6), constrained_layout=True)

for ax, image_db, title in zip(
    axes,
    [focused_uniform_db, focused_hann_db],
    [
        "Focused SAR: uniform aperture weights",
        "Focused SAR: Hann aperture weights",
    ],
):
    image = ax.imshow(
        image_db,
        origin="lower",
        aspect="auto",
        extent=[
            image_cross_range_m[0],
            image_cross_range_m[-1],
            image_downrange_m[0],
            image_downrange_m[-1],
        ],
        cmap="magma",
        vmin=-30,
        vmax=0,
    )

    for _, target in targets.iterrows():
        ax.plot(
            target["cross_range_m"],
            target["downrange_m"],
            marker="x",
            markersize=10,
            markeredgewidth=2.2,
            color="cyan",
        )

    ax.set_xlabel("Azimuth / cross range, $x$ (m)")
    ax.set_ylabel("Downrange, $y$ (m)")
    ax.set_title(title)
    fig.colorbar(image, ax=ax, label="Normalized magnitude (dB)")

plt.show()

### Focused-image interpretation

Both images are focused because both apply the predicted range interpolation
and conjugate propagation phase before summing the aperture.

- **Uniform weighting** uses every position equally. It produces the narrowest
  ideal main lobe, but its sidelobes are higher.
- **Hann weighting** tapers the aperture ends. It suppresses sidelobes but
  broadens the main lobe and reduces unnormalized coherent gain.

The Hann window did not turn an unfocused image into a focused image.
Backprojection already performed the focusing; the window changed the focused
point-spread function.

> **What to say:** Both panels are already focused. Uniform weighting shows
> the narrow response and finite-aperture sidelobes most clearly. Hann
> weighting lowers those sidelobes but broadens each point response. The
> cyan markers are the known target coordinates, not values used by the
> backprojection algorithm.

In [ ]:
# Compare focused azimuth profiles through each target's downrange coordinate.
fig, axes = plt.subplots(1, 2, figsize=(15, 5), constrained_layout=True)

for ax, (_, target) in zip(axes, targets.iterrows()):
    target_downrange_index = int(
        np.argmin(
            np.abs(
                image_downrange_m
                - target["downrange_m"]
            )
        )
    )

    uniform_profile_db = normalized_magnitude_db(
        focused_uniform[target_downrange_index, :],
        floor_db=-50,
    )
    hann_profile_db = normalized_magnitude_db(
        focused_hann[target_downrange_index, :],
        floor_db=-50,
    )

    ax.plot(image_cross_range_m, uniform_profile_db, label="Uniform aperture")
    ax.plot(image_cross_range_m, hann_profile_db, label="Hann aperture")
    ax.axvline(
        target["cross_range_m"],
        color="black",
        linestyle="--",
        label="True azimuth",
    )
    ax.set_xlim(target["cross_range_m"] - 0.12, target["cross_range_m"] + 0.12)
    ax.set_ylim(-45, 3)
    ax.set_xlabel("Azimuth / cross range, $x$ (m)")
    ax.set_ylabel("Normalized magnitude (dB)")
    ax.set_title(
        f"{target['name']} cut at y = {target['downrange_m']:.2f} m"
    )
    ax.legend()

plt.show()

In [ ]:
# Verify that the focused peaks occur near the two known target coordinates.
focused_peak_rows = []
for _, target in targets.iterrows():
    target_downrange_index = int(
        np.argmin(
            np.abs(
                image_downrange_m
                - target["downrange_m"]
            )
        )
    )
    uniform_profile_magnitude = np.abs(
        focused_uniform[target_downrange_index, :]
    )
    local_search_mask = (
        np.abs(
            image_cross_range_m
            - target["cross_range_m"]
        )
        <= 0.050
    )
    local_indices = np.flatnonzero(local_search_mask)
    peak_index = local_indices[
        np.argmax(uniform_profile_magnitude[local_search_mask])
    ]
    estimated_cross_range_m = image_cross_range_m[peak_index]

    focused_peak_rows.append(
        {
            "target": target["name"],
            "true_cross_range_m": target["cross_range_m"],
            "estimated_cross_range_m": estimated_cross_range_m,
            "cross_range_error_mm": 1e3
            * (
                estimated_cross_range_m
                - target["cross_range_m"]
            ),
        }
    )

focused_peak_table = pd.DataFrame(focused_peak_rows)
print(focused_peak_table.to_string(index=False))

## 12. Processing-evolution comparison

The three panels below use different horizontal coordinates and answer
different questions:

1. **Range–angle:** Which physical-array angle aligns the four RX channels at
   the center rail position?
2. **Range–aperture:** What complex range response was measured at each rail
   position before aperture focusing?
3. **Focused SAR:** Which spatial pixel predicts the complete range and phase
   history across the rail?

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(20, 6), constrained_layout=True)

# Panel 1: conventional range-angle map
im0 = axes[0].imshow(
    range_angle_db.T,
    origin="lower",
    aspect="auto",
    extent=[
        candidate_angle_deg[0],
        candidate_angle_deg[-1],
        range_axis_m[0],
        range_axis_m[-1],
    ],
    cmap="viridis",
    vmin=-30,
    vmax=0,
)
axes[0].set_xlim(-20, 20)
axes[0].set_ylim(0.9, 2.05)
axes[0].set_xlabel("Azimuth angle (degrees)")
axes[0].set_ylabel("Range (m)")
axes[0].set_title("1. Conventional range–angle")
fig.colorbar(im0, ax=axes[0], label="dB")

# Panel 2: unfocused range-aperture data
im1 = axes[1].imshow(
    unfocused_range_aperture_db.T,
    origin="lower",
    aspect="auto",
    extent=[
        aperture_positions_m[0],
        aperture_positions_m[-1],
        range_axis_m[0],
        range_axis_m[-1],
    ],
    cmap="magma",
    vmin=-30,
    vmax=0,
)
axes[1].set_ylim(0.9, 2.05)
axes[1].set_xlabel("Azimuth aperture position, $x_a$ (m)")
axes[1].set_ylabel("Range (m)")
axes[1].set_title("2. Unfocused range–aperture data")
fig.colorbar(im1, ax=axes[1], label="dB")

# Panel 3: focused SAR image
im2 = axes[2].imshow(
    focused_hann_db,
    origin="lower",
    aspect="auto",
    extent=[
        image_cross_range_m[0],
        image_cross_range_m[-1],
        image_downrange_m[0],
        image_downrange_m[-1],
    ],
    cmap="magma",
    vmin=-30,
    vmax=0,
)
for _, target in targets.iterrows():
    axes[2].plot(
        target["cross_range_m"],
        target["downrange_m"],
        marker="x",
        markersize=9,
        markeredgewidth=2,
        color="cyan",
    )
axes[2].set_xlabel("Estimated azimuth / cross range, $x$ (m)")
axes[2].set_ylabel("Estimated downrange, $y$ (m)")
axes[2].set_title("3. Focused SAR backprojection")
fig.colorbar(im2, ax=axes[2], label="dB")

plt.show()

## 13. Lecture recap

The detailed teaching script now appears beside the corresponding processing
steps. The three equations below provide a compact closing summary.

**Common range processing**

Common processing:

$$
x[a,f,p,m,n]
\xrightarrow{\mathrm{FFT}_n}
S[a,f,p,m,k]
\xrightarrow{\mathrm{average}_{f,m}}
\overline S[a,p,k].
$$

**Conventional physical-array beamforming**

$$
H[k,u]
\;=\;
\mathbf v^\dagger(\theta_u)\mathbf s_{a_0}[k].
$$

**Focused SAR backprojection**

$$
I[i,j]
\;=\;
\sum_a
w_a[a]
\widetilde S\!\left[a,R_a^{(i,j)}\right]
e^{+j4\pi R_a^{(i,j)}/\lambda}.
$$

## 14. Suggested exercises

1. Move the targets closer together in azimuth/cross range. At what separation does
   the focused image stop showing two distinct peaks?
2. Reduce the synthetic-aperture length while keeping 121 samples. How does the
   azimuth main lobe change?
3. Reduce the number of aperture positions while keeping the same aperture
   length. When do grating lobes or spatial-aliasing artifacts become visible?
4. Reverse the steering-vector exponent sign. How does the conventional angle
   axis change?
5. Replace `steering_matrix.conj().T` with `steering_matrix.T`. Why does the
   physical-array response degrade?
6. Average range-profile magnitudes instead of complex values before
   beamforming. Which products become impossible to form correctly?
7. Give one target a nonzero chirp-to-chirp Doppler phase. Compare simple
   chirp averaging with a Doppler FFT.
8. Extend backprojection to all four RX channels using the exact bistatic path
   $L_{a,p}^{(i,j)}=R_{\mathrm{TX},a}^{(i,j)}+R_{\mathrm{RX},a,p}^{(i,j)}$.